# Perceptron Example (Wine Quality Dataset)

Here it is demonstrated how to use the `Perceptron` module from the CMOR-438 library to perform binary classification.

**Goal: Predict whether a wine is High Quality (score ≥ 7) based on its physicochemical properties.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(REPO_ROOT, 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
# PCA is in src/unsupervised — add that path too for the visualisation cell
SRC_UNSUP = os.path.join(REPO_ROOT, 'src', 'unsupervised')
sys.path.insert(0, SRC_UNSUP)

from perceptron import Perceptron
from pca import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_bin = (wine['quality'].values >= 7).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)
print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train

In [ ]:
p = Perceptron(learning_rate=0.01, n_iterations=300)
p.fit(X_tr, y_tr)
print(f'Accuracy: {p.accuracy(X_te, y_te):.4f}')
print(f'Epochs run: {len(p.errors_per_epoch_)}')
print(f'Final epoch errors: {p.errors_per_epoch_[-1]}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(p.errors_per_epoch_, color='purple', lw=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Misclassifications')
axes[0].set_title('Perceptron Training Errors per Epoch', fontweight='bold')

pca_vis = PCA(n_components=2).fit(X_tr)
X_2d_te = pca_vis.transform(X_te)
preds = p.predict(X_te)
correct = (preds == np.where(y_te==0,-1,1))
axes[1].scatter(X_2d_te[correct,0],  X_2d_te[correct,1],  c='steelblue', s=15, alpha=0.6, label='Correct')
axes[1].scatter(X_2d_te[~correct,0], X_2d_te[~correct,1], c='crimson',   s=20, alpha=0.8, marker='x', label='Wrong')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('Perceptron Errors in PCA Space', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Analysis

**Result: 80.8% test accuracy** — noticeably lower than Logistic Regression's 86.9% on the same task.

The gap illustrates an important distinction: both are linear classifiers, but Logistic Regression minimises a smooth probabilistic loss (Binary Cross-Entropy), while the Perceptron uses a hard mistake-driven update rule. The BCE loss gives Logistic Regression a much stronger training signal — it penalises confident wrong predictions heavily, even when near the boundary.

**Final epoch errors: 157 out of 914 training samples** after 300 epochs. The Perceptron did not converge, which is expected — the wine quality data is not perfectly linearly separable in 11 dimensions, so the Perceptron Convergence Theorem's guarantee does not apply.

**The errors-per-epoch plot** should show a general downward trend that flattens without reaching zero, confirming the algorithm is learning but cannot find a perfect separator.

**The PCA error plot** projects the 11D test data into 2D. Mistakes (red X markers) tend to cluster near the class boundary — these are the ambiguous wines that even more powerful models struggle with.

**Key takeaway:** The Perceptron is a useful pedagogical tool that shows how weight updates work, but it is strictly dominated by Logistic Regression for practical classification. Its value is in understanding the foundations of neural network learning rather than achieving state-of-the-art accuracy.